In [2]:
import torch

In [8]:
tokens_len=7
mask = torch.ones(tokens_len, tokens_len)
for i in range(tokens_len):
    if 1 < i < tokens_len-1:
        mask[i, i-1] = 0
    if 0 < i < tokens_len-2:
        mask[i, i+1] = 0
# mask[0,1:tokens_len] = 0
# mask[1:tokens_len-1,0] = 0
# mask[-1,0:tokens_len-1] = 0
# mask[1:tokens_len-1,-1] = 0
print(mask)

padded_mask = torch.zeros((9, 9))
padded_mask[:tokens_len, :tokens_len] = mask
for i in range(tokens_len, 9):
    
    padded_mask[i,i] = 1
print(padded_mask)

tensor([[1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 0., 1., 1., 1., 1.],
        [1., 0., 1., 0., 1., 1., 1.],
        [1., 1., 0., 1., 0., 1., 1.],
        [1., 1., 1., 0., 1., 0., 1.],
        [1., 1., 1., 1., 0., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1.]])
tensor([[1., 1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 0., 1., 1., 1., 1., 0., 0.],
        [1., 0., 1., 0., 1., 1., 1., 0., 0.],
        [1., 1., 0., 1., 0., 1., 1., 0., 0.],
        [1., 1., 1., 0., 1., 0., 1., 0., 0.],
        [1., 1., 1., 1., 0., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 1.]])


In [ ]:
from transformers import RobertaForSequenceClassification
from parameters import *

model = RobertaForSequenceClassification.from_pretrained(PRETRAINED_MODEL_DIR_8L16HT2, num_labels=2)
print(model.num_parameters())
print(model)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at d:\workspace\iMotifs\train_from_bpe\content\imotifBERT_8L16HT2\pretrained_model and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


15111554
RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(1000, 256, padding_idx=1)
      (position_embeddings): Embedding(128, 256, padding_idx=1)
      (token_type_embeddings): Embedding(1, 256)
      (LayerNorm): LayerNorm((256,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-7): 8 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=256, out_features=256, bias=True)
              (key): Linear(in_features=256, out_features=256, bias=True)
              (value): Linear(in_features=256, out_features=256, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
              (distance_embedding): Embedding(255, 16)
            )
            (output): RobertaSelfOutput(
              (dense): L

In [7]:
from transformers.models.roberta.modeling_roberta import RobertaSdpaSelfAttention
from transformers import T5Config, T5ForSequenceClassification

config = T5Config(
    vocab_size=32128,  # 词汇表大小，T5的小模型是32128
    d_model=512,       # 隐藏层的维度，通常为512
    num_labels=2,      # 分类任务的标签数量，您可以根据需要更改
    num_heads=8,       # 多头注意力机制的头数
    num_layers=4,      # Transformer层数
    d_ff=2048,         # 前馈网络的隐藏层维度
    dropout_rate=0.1   # dropout率
)

# 创建模型
model = T5ForSequenceClassification(config)

# 查看模型结构
print(model)
# 获取 Decoder 层
# decoder = model.decoder

T5ForSequenceClassification(
  (transformer): T5Model(
    (shared): Embedding(32128, 512)
    (encoder): T5Stack(
      (embed_tokens): Embedding(32128, 512)
      (block): ModuleList(
        (0): T5Block(
          (layer): ModuleList(
            (0): T5LayerSelfAttention(
              (SelfAttention): T5Attention(
                (q): Linear(in_features=512, out_features=512, bias=False)
                (k): Linear(in_features=512, out_features=512, bias=False)
                (v): Linear(in_features=512, out_features=512, bias=False)
                (o): Linear(in_features=512, out_features=512, bias=False)
                (relative_attention_bias): Embedding(32, 8)
              )
              (layer_norm): T5LayerNorm()
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (1): T5LayerFF(
              (DenseReluDense): T5DenseActDense(
                (wi): Linear(in_features=512, out_features=2048, bias=False)
                (wo): Linear(in_featu

In [8]:
from transformers import BartConfig, BartForSequenceClassification

# 创建一个自定义的 BART 配置
config = BartConfig(
    vocab_size=50265,         # BART的词汇表大小
    d_model=768,              # Transformer的模型维度
    encoder_layers=12,        # 编码器的层数
    decoder_layers=12,        # 解码器的层数
    attention_heads=12,       # 注意力头数
    hidden_dropout_prob=0.1,  # 隐藏层的 dropout 概率
    attention_probs_dropout_prob=0.1,  # 注意力的dropout概率
    max_position_embeddings=1024,  # 最大位置编码
    d_ff=3072,                # 前馈网络的维度
    num_labels=2              # 用于分类任务的标签数
)

# 用自定义配置初始化模型
model = BartForSequenceClassification(config)

# 打印模型结构
print(model)


BartForSequenceClassification(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50265, 768, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50265, 768, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 768)
      (layers): ModuleList(
        (0-11): 12 x BartEncoderLayer(
          (self_attn): BartSdpaAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=768, bias=True)
          (final_